# Week 6: Validation and Research Claim Audit

**Lane:** CTR / Engagement Opportunity Scoring

This audit tests whether the Week 5 comparison is stable under a split that respects client boundaries and checks whether any claim goes beyond the evidence.

## 1) Two paper findings and my methodology questions

**Finding 1:** The research discussion presents CTR or engagement improvements as useful signals for content action. **Question:** Is the outcome label defined in a future window, or is the same observation window used for both the signal and the claimed improvement?

**Finding 2:** The research discussion treats refresh or optimization signals as associated with better content performance. **Question:** Does the validation design separate clients, pages, or time periods, and therefore support transfer beyond the observed sample?

These are constructive questions about label timing and validation design, not judgments about the paper's value.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

workspace_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'internship-warehouse').exists()), Path.cwd())
root = workspace_root / 'data' / 'internship-warehouse'
con = duckdb.connect()
march = (root / 'fact_content_daily_performance' / 'month=2026-03' / '*.parquet').as_posix()
april = (root / 'fact_content_daily_performance' / 'month=2026-04' / '*.parquet').as_posix()
base = con.sql(f"""SELECT client_hash_id, content_hash_id, SUM(gsc_impressions)::DOUBLE impressions, SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions),0) ctr, SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions),0) avg_position, SUM(sessions_organic)::DOUBLE sessions_organic FROM read_parquet('{march}') WHERE gsc_data_available IS TRUE GROUP BY ALL""").df()
base['expected_ctr'] = 0.10 / base['avg_position'].clip(lower=1).pow(0.35)
base['ctr_gap'] = (base['expected_ctr'] - base['ctr']).clip(lower=0)
future = con.sql(f"""SELECT client_hash_id, content_hash_id, SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions),0) future_ctr FROM read_parquet('{april}') WHERE gsc_data_available IS TRUE GROUP BY ALL""").df()
data = base.merge(future, on=['client_hash_id','content_hash_id']).dropna(subset=['future_ctr','ctr'])
data['label'] = (data['future_ctr'] > data['ctr']).astype(int)
feature_cols = ['impressions','ctr','avg_position','sessions_organic','ctr_gap']
data['group_bucket'] = pd.util.hash_pandas_object(data['client_hash_id'], index=False) % 5
def fit_precision(train, test):
    model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=200, class_weight='balanced'))
    model.fit(train[feature_cols], train.label)
    test = test.copy()
    test['score'] = model.predict_proba(test[feature_cols])[:,1]
    return test.sort_values('score', ascending=False).head(10).label.mean()
group_train, group_test = data[data.group_bucket != 0], data[data.group_bucket == 0]
random_train, random_test = train_test_split(data, test_size=0.2, random_state=42, stratify=data.label)
audit_table = pd.DataFrame({
    'split': ['random row split', 'client-grouped split'],
    'precision_at_10': [fit_precision(random_train, random_test), fit_precision(group_train, group_test)]
})
print(audit_table.to_string(index=False))
print('Error examples from grouped test:')
group_test = group_test.copy()
model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=200, class_weight='balanced'))
model.fit(group_train[feature_cols], group_train.label)
group_test['score'] = model.predict_proba(group_test[feature_cols])[:,1]
group_test['error'] = np.select([group_test.score >= 0.5, group_test.label.eq(1)], ['false_positive','false_negative'], default='correct_negative')
print(group_test[group_test.error != 'correct_negative'][['client_hash_id','content_hash_id','score','label','error']].head(5).to_string(index=False))

               split  precision_at_10
    random row split              0.3
client-grouped split              0.3
Error examples from grouped test:
         client_hash_id          content_hash_id    score  label          error
client_3197e6291363b4db content_3254c9380fa0af8f 0.511606      0 false_positive
client_3197e6291363b4db content_0e4cfb8d42d457de 0.504493      0 false_positive
client_3197e6291363b4db content_c711a43ee1459d9e 0.525848      0 false_positive
client_3197e6291363b4db content_8e6f57489e905fed 0.511344      0 false_positive
client_3197e6291363b4db content_79f6c715e42ad41c 0.500837      0 false_positive


## 2) My model under an honest split

The random split is an optimistic diagnostic because the same client's behavioral patterns can appear in train and test. The client-grouped result is the decision-relevant audit number. It is the design I will carry forward.

## 3) Leakage audit

- Allowed: March impressions, CTR, average position, organic sessions, and the March CTR gap.
- Excluded: April clicks, April impressions, April CTR, and the future label itself.
- Group boundary: all rows for a client stay in one split.
- Public safety: pseudonymous hash keys are not interpreted as client or content identities.

## 4) Claim rewrite

Overclaim: **The Logistic Regression model predicts which pages will improve.**

Evidence-matched claim: **On this March-to-April slice, the Logistic Regression scorer produced a measured Precision@10 on held-out clients; the Week 4 baseline was higher on that metric. The result is directional decision-support evidence, not a causal claim or production guarantee.**

## 5) Self-check

The notebook names two methodology questions, compares random and client-grouped validation, audits leakage, shows error examples, and uses observed/measured/directional claim language.